# Week 1: Classical Hamming [7,4,3] hard and soft decoding

This notebook explains the package implementation; it does not duplicate the algorithms. Run it after `python -m pip install -e .[dev]` from the repository root.

In [ ]:
import numpy as np
from softqec import gf2
from softqec.classical_channels import bpsk_modulate, hard_demodulate
from softqec.classical_codes import HammingCode74
from softqec.classical_decoders import hard_syndrome_decode, exact_soft_ml_decode

code = HammingCode74()
print('G =\n', code.generator)
print('H =\n', code.parity_check)
print('H G^T =\n', gf2.matmul(code.parity_check, code.generator.T))

## Enumerate and validate the codebook

A linear [7,4] code contains 16 codewords. Its minimum distance is the smallest nonzero codeword weight.

In [ ]:
messages = code.messages()
codewords = code.codewords()
print('number of codewords:', len(codewords))
print('minimum distance:', code.minimum_distance())
np.column_stack([messages, codewords])

## One hard-decision example

A one-bit error produces the corresponding column of H as its syndrome.

In [ ]:
u = np.array([1, 0, 1, 1], dtype=np.uint8)
c = code.encode(u)
r = c.copy(); r[2] ^= 1
decoded, correction = hard_syndrome_decode(r, code)
print('message:   ', u)
print('codeword:  ', c)
print('received:  ', r)
print('syndrome:  ', code.syndrome(r))
print('correction:', correction)
print('decoded:   ', decoded)

## Why soft information can help

The next observation has two slightly negative components. Hard thresholding records two bit flips because it keeps only their signs and has no representation of confidence. Exact ML sees that both values are very close to zero and compares the whole reliability pattern against every valid codeword.

In [ ]:
observation = bpsk_modulate(codewords[0])
observation[[0, 1]] = [-0.05, -0.05]
hard_word, _ = hard_syndrome_decode(hard_demodulate(observation), code)
soft_word = exact_soft_ml_decode(observation, code)
print('observation:', observation)
print('hard word:  ', hard_word)
print('soft word:  ', soft_word)

## Full experiment

Generate the saved result table and figure from the terminal with `make classical`. Interpret Monte Carlo points together with their Wilson intervals and retain the configuration and seed metadata.